In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00


In [ ]:
%%writefile eval_kaggle_single.py
import os
import json
import torch
import random
import numpy as np
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from tqdm import tqdm

# ---------------- CONFIG -----------------
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Hugging Face login
login(token="hf_token")

# Model paths
BASE_MODEL = "meta-llama/Llama-2-7b-chat-hf"
ADAPTER_MODEL = "pradip777/llama-2-PCG"

# Dataset and output
DATASET_FILE = "/kaggle/input/datasets/pradippokhrel77/test-datasets/refined_test.jsonl"
OUTPUT_FILE = "/kaggle/working/adapted_chatmodel_codes.json"

# Generation parameters
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.9
REPETITION_PENALTY = 1.2
SAVE_EVERY = 5  # save results after every N prompts

# ---------------- DEVICE -----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- LOAD TOKENIZER -----------------
print("🚀 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

# ---------------- LOAD BASE MODEL -----------------
print("🚀 Loading base model with 4-bit quantization (if GPU available)...")
if DEVICE == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    device_map = "auto"
else:
    bnb_config = None
    device_map = {"": "cpu"}

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map=device_map
)

# ---------------- LOAD LoRA ADAPTER -----------------
print("🚀 Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL)
model.eval()
print(f"✅ Model loaded on {DEVICE}!")

# ---------------- CODE EXTRACTION -----------------
def extract_code(model_output: str) -> str:
    """Extract clean Python code from model output"""
    if '[/INST]' in model_output:
        model_output = model_output.split('[/INST]')[-1].strip()
    
    # Extract code between triple backticks
    match = re.search(r'```(?:python)?\s*\n(.*?)\n```', model_output, flags=re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Fallback: lines starting with code keywords
    code_lines = []
    for line in model_output.splitlines():
        if line.strip().startswith(("def ", "import ", "class ", "if ", "for ", "while ")):
            code_lines.append(line)
    return "\n".join(code_lines).strip() or "NONE"

# ---------------- GENERATE CODE -----------------
def generate_code(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# ---------------- PROCESS DATASET -----------------
results = []
processed_prompts = set()

# Load existing partial results to resume
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        results = json.load(f)
        processed_prompts = {r["prompt"] for r in results}

if not os.path.exists(DATASET_FILE) or os.stat(DATASET_FILE).st_size == 0:
    raise FileNotFoundError(f"Dataset missing or empty: {DATASET_FILE}")

# Read JSONL dataset
problems = []
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                problems.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"⚠️ JSON decode error: {e} | line: {line[:100]}")

print(f"✅ Loaded {len(problems)} problems from dataset")

# Generate code for each problem
for idx, problem in enumerate(tqdm(problems, desc="Generating code")):
    prompt_text = problem.get("prompt", "")
    if prompt_text in processed_prompts:
        continue  # Skip already processed prompts
    
    model_output = generate_code(prompt_text)
    code = extract_code(model_output)
    
    results.append({
        "prompt": prompt_text,
        "reference": problem.get("code", "NONE"),
        "model_output": model_output,
        "code": code
    })
    
    processed_prompts.add(prompt_text)

    # Logging progress
    print(f"✅ [{idx+1}/{len(problems)}] Generated code for prompt: {prompt_text[:50]}...")

    # Incremental save every SAVE_EVERY prompts
    if (idx + 1) % SAVE_EVERY == 0 or (idx + 1) == len(problems):
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        print(f"💾 Saved {len(results)} results to {OUTPUT_FILE}")

print(f"🎉 Done! All results saved to {OUTPUT_FILE}")

Writing eval_kaggle_single.py


In [3]:
!python /kaggle/working/eval_kaggle_single.py

🚀 Loading tokenizer...
config.json: 100%|█████████████████████████████| 614/614 [00:00<00:00, 3.05MB/s]
tokenizer_config.json: 100%|███████████████| 1.62k/1.62k [00:00<00:00, 8.27MB/s]
tokenizer.json: 100%|██████████████████████| 1.84M/1.84M [00:00<00:00, 14.4MB/s]
tokenizer.model: 100%|███████████████████████| 500k/500k [00:00<00:00, 1.44MB/s]
special_tokens_map.json: 100%|█████████████████| 414/414 [00:00<00:00, 2.06MB/s]
🚀 Loading base model with 4-bit quantization (if GPU available)...
model.safetensors.index.json: 100%|████████| 26.8k/26.8k [00:00<00:00, 10.3MB/s]
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:39<00:00, 19.58s/it]
Download complete: 100%|████████████████████| 13.5G/13.5G [00:39<00:00, 343MB/s]
Loading weights: 100%|█| 291/291 [00:04<00:00, 65.75it/s, Materializing param=mo
generation_config.json: 100%|███████████████████| 188/188 [00:00<00:00, 730kB/s]
🚀 Loading LoRA adapter...
adapter_config.json: 1.07kB [00:00, 582kB/s]
adapter_model.safetensors: 10